### Import packages, define paths and load data

In [199]:
from pathlib import Path
import json
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import re
import numpy as np

/Users/tildeidunsloth/Desktop/Thesis/Thesis_project/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [182]:
# Define project and data paths
PROJECT_ROOT = Path("/Users/tildeidunsloth/Desktop/Thesis")
DATA_DIR = PROJECT_ROOT / "data/cleaned"

sci_fi_data_path = DATA_DIR / "sci_fi_stories_cleaned.jsonl"
romance_data_path = DATA_DIR / "romance_stories_cleaned.jsonl"
literary_fiction_data_path = DATA_DIR / "lit_fiction_stories_cleaned.jsonl"

In [183]:
def load_jsonl(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

In [184]:
lit_fic_stories = load_jsonl(literary_fiction_data_path)
romance_stories = load_jsonl(romance_data_path)
sci_fi_stories = load_jsonl(sci_fi_data_path)

In [185]:
# merge all stories into one list
all_stories = lit_fic_stories + romance_stories + sci_fi_stories

In [186]:
# convert jsonl to pandas dataframe
df = pd.DataFrame(all_stories)

In [187]:
def preprocess(text):
    
    # remove words starting with a capital letter (likely names)
    text = re.sub(r'\b[A-Z][a-z]*\b', '', text)
    
    # remove punctuation and numbers
    text = re.sub(r"[^a-z\s]", "", text)
    
    return text

In [188]:
df = pd.DataFrame(all_stories)

In [189]:
df["clean_story"] = df["story"].apply(preprocess)

In [190]:
custom_stopwords = ['engineer', 'architect', 'pilot', 'construction', 'barber', 'nurse', 'librarian', 'assistant', 'teacher', 'psychologist', 'artist', 'musician', 'traveler', 'character', 'local', 'visitor', 'person', 'vacation', 'christmas', 'payday', 'zoo', 'park', 'home', 'summer', 'prison', 'building', 'funeral', 'war', 'confrontation', 'courtroom', 'crime', 'school', 'cafeteria', 'winter', 'casino', 'office', 'hallway', 'hostel', 'asked', 'walked', 'watched', 'thought', 'want']

In [191]:
all_stopwords = list(ENGLISH_STOP_WORDS.union(custom_stopwords))

In [192]:
vectorizer = TfidfVectorizer(
    stop_words=all_stopwords,
    min_df=10,
    max_df=0.85
)
X = vectorizer.fit_transform(df["clean_story"])

In [193]:
lda = LatentDirichletAllocation(
    n_components=10,
    random_state=42,
    n_jobs=-1
)

lda.fit(X)

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",10
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


In [194]:
words = vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(lda.components_):
    top_words = [words[i] for i in topic.argsort()[-10:]]
    print(f"Topic {topic_idx}: {top_words}")

Topic 0: ['blacklisted', 'institutes', 'ruthlessly', 'residuals', 'reissued', 'societal', 'recruiting', 'renaissance', 'causal', 'confit']
Topic 1: ['regenerative', 'piezo', 'arboretum', 'pangolins', 'fungal', 'newsfeed', 'entities', 'blacklisted', 'residuals', 'societal']
Topic 2: ['regenerative', 'piezo', 'arboretum', 'pangolins', 'fungal', 'newsfeed', 'entities', 'blacklisted', 'residuals', 'societal']
Topic 3: ['imply', 'dampening', 'catwalk', 'willful', 'shined', 'unevenness', 'mangled', 'trespassers', 'rodent', 'fob']
Topic 4: ['regenerative', 'piezo', 'arboretum', 'pangolins', 'fungal', 'newsfeed', 'entities', 'blacklisted', 'residuals', 'societal']
Topic 5: ['bliss', 'blacklisted', 'residuals', 'sedative', 'sophisticated', 'disturbingly', 'societal', 'euthanasia', 'manganese', 'macaque']
Topic 6: ['feline', 'reptiles', 'subsumed', 'derail', 'infrared', 'rabbits', 'glaciers', 'reacts', 'shorebirds', 'leopard']
Topic 7: ['regenerative', 'piezo', 'arboretum', 'pangolins', 'fungal'

In [195]:
topic_distribution = lda.transform(X)

In [196]:
topic_df = pd.DataFrame(
    topic_distribution,
    columns=[f"topic_{i}" for i in range(10)]
)

df = pd.concat([df, topic_df], axis=1)

In [197]:
df

,story,genre,story_hint,agent,agent_type,event,event_valence,context,context_valence,model,...,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9
0,The noon crowd in the cafeteria had the kind o...,literary fiction,A librarian makes an announcement in a cafeteria,A librarian,occupational_feminine_stereotyped,makes an announcement,neutral,in a cafeteria,neutral,gpt-5-mini,...,0.004947,0.004947,0.004947,0.004947,0.004947,0.004947,0.004947,0.004947,0.004948,0.955479
1,"On the first morning of summer break, Claire u...",literary fiction,A librarian observes the surroundings during t...,A librarian,occupational_feminine_stereotyped,observes the surroundings,neutral,during the summer break,positive,gpt-5-mini,...,0.004746,0.004746,0.004746,0.004746,0.004746,0.004746,0.004746,0.004746,0.004748,0.957280
2,Maren had never liked zoos. The first time she...,literary fiction,A nurse is afraid at the zoo,A nurse,occupational_feminine_stereotyped,is afraid,negative,at the zoo,positive,gpt-5-mini,...,0.004999,0.004999,0.004999,0.004999,0.004999,0.004999,0.004999,0.004999,0.005001,0.955004
3,On the morning the snow began to make ordinary...,literary fiction,A character abandons their friend during chris...,A character,non_occupational,abandons their friend,negative,during christmas time,positive,gpt-5-mini,...,0.004911,0.004911,0.004911,0.004911,0.004911,0.004911,0.004911,0.004911,0.004912,0.955802
4,Elias kept his scissors close the way some men...,literary fiction,A barber loses something valuable at the zoo,A barber,occupational_masculine_stereotyped,loses something valuable,negative,at the zoo,positive,gpt-5-mini,...,0.005577,0.005577,0.005577,0.005577,0.005577,0.005577,0.005577,0.005577,0.005578,0.949808
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17995,The Vault of Lusa sat like a beacon at the piv...,science fiction,A visitor achieves a lifelong dream during a c...,A visitor,non_occupational,achieves a lifelong dream,positive,during a confrontation,negative,gpt-5-mini,...,0.004679,0.004679,0.004679,0.004679,0.004679,0.004679,0.004679,0.004679,0.957884,0.004681
17996,Maris carried the bag like a rebellion.\n\nIt ...,science fiction,An artist carries a bag in a time of war,An artist,occupational_neutral,carries a bag,neutral,in a time of war,negative,gpt-5-mini,...,0.005067,0.005067,0.005067,0.005067,0.005067,0.005067,0.005067,0.005067,0.573743,0.385720
17997,The first week of winter at the Halley Researc...,science fiction,A psychologist discovers something new during ...,A psychologist,occupational_feminine_stereotyped,discovers something new,positive,during winter,neutral,gpt-5-mini,...,0.004455,0.004455,0.004455,0.004455,0.004455,0.004455,0.004455,0.004455,0.959900,0.004457
17998,Mara kept her instrument in the corner where t...,science fiction,A musician is sad in their home,A musician,occupational_neutral,is sad,negative,in their home,positive,gpt-5-mini,...,0.004793,0.004793,0.004793,0.004793,0.004793,0.004793,0.004793,0.004793,0.728070,0.233585
